# Build a `MarketContext` from raw quotes

**Purpose:** Connect market-data foundations to the calibration engine by turning raw quote envelopes into a reusable `MarketContext`.

**Prerequisites:** `01_foundations/market_data_and_curves.ipynb` and `02_pricing/pricing_fundamentals.ipynb`.

**What you'll learn:**

- Load reference calibration envelopes from JSON files.
- Run `calibrate(envelope_json)` and inspect the returned `CalibrationResult`.
- Persist calibrated markets and use diagnostics before solving.

This notebook walks through the analyst-facing flow for building a `MarketContext` from a JSON `CalibrationEnvelope`. It is the canonical entry point: `calibrate(envelope_json).market`.


In [ ]:
import json
from pathlib import Path

from finstack_quant.valuations import calibrate

# Path to the reference envelope. Adjust if running outside the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("could not locate the repo root from cwd")
    REPO_ROOT = REPO_ROOT.parent

envelope_path = (
    REPO_ROOT
    / "finstack-quant"
    / "valuations"
    / "examples"
    / "market_bootstrap"
    / "01_usd_discount.json"
)
envelope_json = envelope_path.read_text()
envelope = json.loads(envelope_json)
print(f"schema: {envelope['schema']}")
print(f"plan id: {envelope['plan']['id']}")
print(f"steps: {[step['id'] for step in envelope['plan']['steps']]}")

In [ ]:
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"rmse: {result.rmse:.3e}")
print(f"max residual: {result.max_residual:.3e}")
print(f"iterations: {result.iterations}")
result.report_to_dataframe()

In [ ]:
# `result.market` is the calibrated MarketContext, ready for downstream use.
market = result.market

# Look up the discount curve we just built.
curve = market.get_discount("USD-OIS")
print(f"discount factor at t=0:  {curve.df(0.0):.6f}")
print(f"discount factor at t=1y: {curve.df(1.0):.6f}")
print(f"discount factor at t=5y: {curve.df(5.0):.6f}")

## Persisting the materialized market

`result.market_json` returns the same context as a JSON snapshot. This is
useful for caching a calibrated market between processes or for diff'ing
two calibrated states.

`MarketContext` round-trips through this snapshot losslessly:
deserialize via `MarketContext.from_json(...)` (Python) or
`MarketContext::try_from(state)` (Rust).

In [ ]:
# Round-trip via the materialized JSON snapshot.
snapshot_json = result.market_json
print(f"snapshot length: {len(snapshot_json)} bytes")
# Phase 2 of the notebook tour will demonstrate composing markets and
# pulling FX cross rates / bond prices / equity spots from market_data
# (the v3 envelope's flat input list).

## Compose markets — chained envelope

An analyst's morning bootstrap typically chains discount → hazard → base correlation
in a single envelope. `12_full_credit_desk_market.json` shows the full pattern: rates
and CDS quotes feed two calibration steps, and a tranche quote_set drives the third.
FX cross rates ride along in `market_data` as `fx_spot` entries since they're
snapshot-only data.

In [ ]:
envelope_path = REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / "12_full_credit_desk_market.json"
envelope_json = envelope_path.read_text()
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"steps: {result.step_ids}")
print(f"rmse: {result.rmse:.3e}")
result.report_to_dataframe()

In [ ]:
import json

market = result.market
discount = market.get_discount("USD-OIS")
hazard = market.get_hazard("CDX-NA-IG-46")

# Base-correlation curves are stored in the market snapshot; reconstruct via the type directly.
from finstack_quant.core.market_data import BaseCorrelationCurve
snapshot = json.loads(market.to_json())
bc_data = next(c for c in snapshot["curves"] if c["type"] == "base_correlation")
bc = BaseCorrelationCurve(id=bc_data["id"], knots=list(zip(bc_data["detachment_points"], bc_data["correlations"])))

print(f"USD-OIS DF(5y):    {discount.df(5.0):.6f}")
print(f"CDX-IG-46 SP(5y):  {hazard.sp(5.0):.6f}")
print(f"BC at 7%:          {bc.correlation(7.0):.4f}")

## Snapshot-only data — FX, bonds, equities

FX matrices, bond prices, equity spots, and dividend schedules are not bootstrapped
today — they ride in `market_data` as `fx_spot`, `price`, and `dividend_schedule`
entries. The reference envelopes 09 / 10 / 11 demonstrate each pattern.

In [ ]:
for name in ["09_fx_matrix.json", "10_bond_prices.json", "11_equity_spots_dividends.json"]:
    path = REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / name
    result = calibrate(path.read_text())
    print(f"{name}: success={result.success}, steps={result.step_ids}")

In [ ]:
# Bond prices are snapshot-only and live in the market JSON under the 'prices' key.
bond_envelope = (REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / "10_bond_prices.json").read_text()
bond_market = calibrate(bond_envelope).market
prices = json.loads(bond_market.to_json()).get("prices", {})
for bond_id in ["US-TREASURY-10Y-2026-05-08", "IBM-7YR-2033"]:
    p = prices[bond_id]["price"]
    print(f"{bond_id}: {p['amount']} {p['currency']}")

## Validate before solving

Phase 4 will add `dry_run(envelope_json)` for fast pre-flight validation
(missing dependencies, undefined quote sets, quote class mismatches) — running
structural checks in microseconds before the (much slower) solver. For now,
`validate_calibration_json` returns the canonical pretty-printed JSON and
surfaces serde-level errors; it is the only validation tool until Phase 4 lands.

In [ ]:
from finstack_quant.valuations import validate_calibration_json
envelope_path = REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / "01_usd_discount.json"
canonical = validate_calibration_json(envelope_path.read_text())
print(canonical[:200] + "...")

## When envelopes fail

Phase 4 surfaces structured diagnostics for envelope failures. Two cheap pre-flight tools are `dry_run` (runs all structural checks without invoking the solver — microseconds) and `dependency_graph_json` (dumps the step DAG). Both raise `CalibrationEnvelopeError` on malformed JSON; structural failures surface as entries in `dry_run`'s report.

The cell below deliberately misspells a `quote_set` reference. `dry_run` catches it and points at the step + suggests a fix. `calibrate` would produce the same error message at runtime — but `dry_run` is faster and lets the analyst fix the envelope before the slow solve.

In [ ]:
from finstack_quant.valuations import dry_run, dependency_graph_json, CalibrationEnvelopeError

envelope_path = REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / "01_usd_discount.json"
broken = json.loads(envelope_path.read_text())

# Find the step's actual quote_set name and deliberately corrupt it.
original_qs = broken["plan"]["steps"][0]["quote_set"]
broken["plan"]["steps"][0]["quote_set"] = original_qs + "s"  # extra 's'

report = json.loads(dry_run(json.dumps(broken)))
for err in report["errors"]:
    if err["kind"] == "undefined_quote_set":
        print(f"[{err['kind']}] step '{err['step_id']}': quote_set '{err['ref_name']}' not found. "
              f"Suggestion: '{err.get('suggestion')}'")
    else:
        print(f"[{err['kind']}] {err}")

### Catching the typed exception

If a real failure happens downstream (e.g., solver non-convergence), `CalibrationEnvelopeError` carries `kind`, `step_id`, and `details` for programmatic handling. It inherits from `RuntimeError`, so existing `except RuntimeError` catchers continue to work.

In [ ]:
# Demonstrate the typed exception by parsing intentionally malformed JSON.
try:
    dry_run("{ malformed")
except CalibrationEnvelopeError as exc:
    print(f"caught {exc.__class__.__name__}: kind={exc.kind}, step={exc.step_id}")
    payload = json.loads(exc.details)
    print(f"details kind: {payload['kind']}")

### Inspecting the dependency graph

`dependency_graph_json` returns the static DAG of a plan's steps — useful for visualizing layered envelopes (envelope 12) or debugging missing dependencies.

In [ ]:
envelope_path = REPO_ROOT / "finstack-quant" / "valuations" / "examples" / "market_bootstrap" / "12_full_credit_desk_market.json"
graph = json.loads(dependency_graph_json(envelope_path.read_text()))
print(f"initial_ids ({len(graph['initial_ids'])}): {graph['initial_ids']}")
for node in graph["nodes"]:
    reads = ", ".join(node["reads"]) or "(none)"
    writes = ", ".join(node["writes"]) or "(none)"
    print(f"step[{node['step_index']}] {node['step_id']} ({node['kind']}): "
          f"reads [{reads}] -> writes [{writes}]")

## Constructing envelopes with TypedDicts

Phase 5 ships `TypedDict` definitions in `finstack_quant.valuations.envelope` for analysts who'd rather build envelopes with editor autocomplete than edit JSON. They have no runtime overhead — they're plain dicts with documented shape — and they round-trip through `calibrate` / `dry_run` verbatim.

Coverage in v1: `CalibrationEnvelope`, `CalibrationPlan`, the four most common step kinds (`discount`, `forward`, `hazard`, `vol_surface`), and the most common rate / CDS quote variants. Other variants fall through as `dict[str, Any]` — combine with `dry_run` (Phase 4) for structural checks that catch typos.

In [ ]:
from finstack_quant.valuations import (
    CalibrationEnvelope,
    CalibrationPlan,
    DiscountStep,
    MarketDatum,
    RateDeposit,
    RateSwap,
)

# Quotes — the editor narrows on `class` and `type` literals. These are
# v2-style `MarketQuote` TypedDicts that ride into the v3 envelope as
# `market_data` entries with a `"kind"` discriminator.
deposit: RateDeposit = {
    "class": "rates",
    "type": "deposit",
    "id": "USD-SOFR-DEP-3M",
    "index": "USD-SOFR-OIS",
    "pillar": {"tenor": {"count": 3, "unit": "months"}},
    "rate": 0.052,
}
swap: RateSwap = {
    "class": "rates",
    "type": "swap",
    "id": "USD-OIS-SWAP-1Y",
    "index": "USD-SOFR-OIS",
    "pillar": {"tenor": {"count": 1, "unit": "years"}},
    "rate": 0.051,
}

# In the v3 envelope, quotes live in the flat `market_data` list keyed by
# `"kind"`; `plan.quote_sets` is just a named list of QuoteIds that resolve
# back into `market_data`.
market_data: list[MarketDatum] = [
    {"kind": "rate_quote", **{k: v for k, v in deposit.items() if k != "class"}},
    {"kind": "rate_quote", **{k: v for k, v in swap.items()    if k != "class"}},
]

step: DiscountStep = {
    "id": "USD-OIS",
    "quote_set": "usd_quotes",
    "kind": "discount",
    "curve_id": "USD-OIS",
    "currency": "USD",
    "base_date": "2026-05-08",
}

plan: CalibrationPlan = {
    "id": "usd_curves_typed",
    "quote_sets": {"usd_quotes": [deposit["id"], swap["id"]]},
    "steps": [step],
}
typed_envelope: CalibrationEnvelope = {
    "schema": "finstack_quant.calibration",
    "plan": plan,
    "market_data": market_data,
}

# Pre-flight check, then calibrate.
report = json.loads(dry_run(json.dumps(typed_envelope)))
print(f"errors: {report['errors']}")
result = calibrate(json.dumps(typed_envelope))
print(f"success: {result.success}, rmse: {result.rmse:.3e}")

## Takeaways

- Calibration envelopes keep raw market quotes outside notebook code while preserving a reproducible market-building workflow.
- `calibrate` returns a `CalibrationResult`; inspect residuals and `result.market` before downstream pricing.
- Use `dry_run` and `dependency_graph_json` for fast diagnostics before invoking slower calibration steps.

**Next:** Use the calibrated `MarketContext` in `02_pricing/pricing_fundamentals.ipynb`.


## Typed calibration result

The object returned by `calibrate` is a `CalibrationResult`: a typed wrapper with residual diagnostics, step reports, and the calibrated `MarketContext`.

In [ ]:
from finstack_quant.valuations import CalibrationResult

print("CalibrationResult type:", CalibrationResult.__name__)
print("latest result is CalibrationResult:", isinstance(result, CalibrationResult))
print("available step ids:", result.step_ids)